# 16.3 기말 프로젝트 리뷰 — 학기의 검증 절차를 한 흐름으로 끝까지 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter16_3_final_project_review.ipynb)

책 본문: [16.3 ML1 총정리와 ML2로 가는 길](https://smhanlab.com/book-ml/kor/ml1/chapter16/3.html)

이 절은 캡스톤 절(기말 프로젝트 채점 + peer-review 결실 + 학기 총정리)이라,
노트북은 새 개념을 소개하지 않는다. 대신 이 학기에 가장 많이 쓴
**검증 절차 — 3분할 → train으로만 fit → val로 순위 → 베이스라인 →
test 딱 한 번** — 을 breast_cancer(569×30)로 한 흐름에 걸쳐 실행한다.
그리고 보고서가 "정직하지 않게" 만들어지는 세 가지 — 선택 편향(§6),
val/test 순위 일치 비보증(§7), 눈에 안 보이는 누수(§8) — 를 숫자로
보여준다. scikit-learn만 쓰고, 모든 시드는 고정이다.
(Colab에서는 첫 셀의 `IMG` 경로를 `/tmp`로 바꾸면 됩니다.)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import os
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

## 1. 문제 정의: "무엇을, 어떤 라벨로 예측하는가"

16.1절 프로젝트의 첫 단계와 똑같이, 모델보다 먼저 **문제를** 정한다.
여기는 8.1절 미니 예시와 같은 breast_cancer를 쓴다 — 행(row)은
**환자** 하나, 라벨은 **양성(1)/악성(0)**, 주 지표는 **F1**(Ch02.3:
클래스가 약 6:4로 기울어져 있어 정확도보다 F1). 이 세 줄(행의 단위·
라벨·지표)이 보고서의 "문제 정의" 절 전체다.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
print(f"data: {X.shape[0]}환자 x {X.shape[1]}특성, 양성 비율 {y.mean():.3f}")

# 3분할: train 60% / val 20% / test 20% (Ch06.3, stratify, seed 0)
X1, Xte, y1, yte = train_test_split(X, y, test_size=0.20, random_state=0, stratify=y)
Xtr, Xva, ytr, yva = train_test_split(X1, y1, test_size=0.25, random_state=0, stratify=y1)
print(f"splits: train {len(Xtr)} / val {len(Xva)} / test {len(Xte)}  (train 양성 비율 {ytr.mean():.3f})")

data: 569환자 x 30특성, 양성 비율 0.627
splits: train 341 / val 114 / test 114  (train 양성 비율 0.628)


## 2. 전처리: 스케일러는 train으로만 fit (Ch06.3)

30개 특성의 스케일이 달라(Ch04, Ch05의 거리 기반 모델) 표준화를 한다.
**핵심 원칙**: `fit`은 train으로만. §8에서 "전체 데이터로 fit하면"이
이 깨끗한 데이터에서는 *어떤 차이도 안 보일 수 있는지*를 직접 확인한다.

In [3]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(Xtr)          # <- fit은 train으로만
Xtr_s, Xva_s, Xte_s = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
print(f"scale_ (train으로 fit): min {sc.scale_.min():.3f}, max {sc.scale_.max():.3f}")

scale_ (train으로 fit): min 0.002, max 592.030


## 3. 모델 후보 5개: Ch02/04/07.1/07.2/07.3 — val F1로 순위 매기기

기말 프로젝트(16.1)는 Ch09~15 딥러닝 기법 최소 1개를 요구하지만,
**절차 자체는 모델 종류와 무관하다**. 그래서 여기서는 Block A의
대표 모델 5개로 절차 전체를 시연한다 — (1) 로지스틱회귀(Ch02, 선형),
(2) kNN(Ch04, 거리), (3) 결정트리(Ch07.1), (4) 랜덤포레스트(Ch07.2),
(5) GBDT(Ch07.3). 하이퍼파라미터는 고정(튜닝은 8.1절에서 이미
보여줬으므로), **val F1로만** 순위 매긴다.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score

models = {
    "logreg(Ch02)":        LogisticRegression(C=1.0, max_iter=2000),
    "kNN k=5(Ch04)":       KNeighborsClassifier(n_neighbors=5),
    "tree d=3(Ch07.1)":    DecisionTreeClassifier(max_depth=3, random_state=0),
    "RF(Ch07.2)":          RandomForestClassifier(n_estimators=100, max_depth=3, random_state=0, n_jobs=-1),
    "GBDT(Ch07.3)":        GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=0),
}
rows = []
for name, m in models.items():
    m.fit(Xtr_s, ytr)                       # <- train으로만 fit
    fva = f1_score(yva, m.predict(Xva_s))   # <- 순위는 val에서만
    rows.append([name, fva, m])
rows.sort(key=lambda r: -r[1])
print(f"{'모델':20s} val F1")
for name, fva, _ in rows:
    print(f"{name:20s} {fva:.3f}")
print("-> val 1위:", rows[0][0])

모델                   val F1
logreg(Ch02)         0.993
kNN k=5(Ch04)        0.979
RF(Ch07.2)           0.957
GBDT(Ch07.3)         0.950
tree d=3(Ch07.1)     0.944
-> val 1위: logreg(Ch02)


## 4. 베이스라인: "무엇도 배우지 않은" 모델 (Ch02.3)

모델 성능을 논하기 전에 **제로점**부터 — train의 다수 클래스만 전부
예측하는 모델. 이 숫자를 넘지 못하면 "모델을 만들었다"고 말할 수
없다(8.1절).

In [5]:
maj = int(np.mean(ytr) >= 0.5)                 # 다수 클래스 (train 정보만 사용)
base_val_f1  = f1_score(yva, np.full(len(yva), maj))
base_val_acc = accuracy_score(yva, np.full(len(yva), maj))
print(f"다수 클래스 = {maj}(악성), 베이스라인 val F1 = {base_val_f1:.3f}, val acc = {base_val_acc:.3f}")
print(f"val 1위 모델과의 격차: {rows[0][1] - base_val_f1:+.3f} (F1)")

다수 클래스 = 1(악성), 베이스라인 val F1 = 0.768, val acc = 0.623
val 1위 모델과의 격차: +0.225 (F1)


## 5. test 측정 — 딱 한 번 (Ch06.3)

val F1로 고른 **최종 1개 모델만** test에 적용한다. 이 숫자가 이
미니-프로젝트의 "최종 성능"이다 — 여기서부터는 이 모델의 test
성능을 다시 보거나, test를 봐서 다른 결정을 하는 것이 금지된다.

In [6]:
final_name, final_f1val, final = rows[0]
acc_te = accuracy_score(yte, final.predict(Xte_s))
f1_te  = f1_score(yte, final.predict(Xte_s))
print(f"최종 모델: {final_name}")
print(f"test accuracy = {acc_te:.3f}, test F1 = {f1_te:.3f}")
print(f"(베이스라인 val F1 {base_val_f1:.3f} 대비, val F1 {final_f1val:.3f})")

최종 모델: logreg(Ch02)
test accuracy = 0.974, test F1 = 0.979
(베이스라인 val F1 0.768 대비, val F1 0.993)


## 6. "test를 살짝 훔쳐봤다면"? — 선택 편향을 숫자로 (Ch06.3)

만약 5개(또는 50개) 후보를 **test**로 돌려보고 제일 좋은 것을 골랐다면?
각 후보의 test 성능 = (진짜 성능 + 잡음)으로 모델링하면, 후보 \(m\)개
중 max를 고르는 행위는 잡음 \(m\)번 뽑기 중 최고를 고르는 것이다.
잡음 \(\sigma=0.02\)(F1 스케일)로 시뮬레이션한다 — 8.1절
§"test를 한 번만"의 숫자를 그대로 재확인한다.

In [7]:
rng = np.random.default_rng(1)
for m in [1, 5, 50]:
    bias = rng.normal(0, 0.02, size=(20000, m)).max(axis=1).mean()
    print(f"후보 {m:>2d}개: E[max] \u2248 {bias:+.4f}")
print("-> '50개 후보 중 test 최고를 고르면' F1이 평균 0.045 부풀어 오른다.")

후보  1개: E[max] ≈ -0.0002
후보  5개: E[max] ≈ +0.0232
후보 50개: E[max] ≈ +0.0449
-> '50개 후보 중 test 최고를 고르면' F1이 평균 0.045 부풀어 오른다.


## 7. val 순위 vs test 순위 — "같다"는 보장이 아니다

정직한 절차(val로 고르기)를 따르면 test 순위를 *예측*할 수 없다 —
그것이 test를 한 번만 쓰는 이유의 다른 면이다. 이 데이터의 val
순위와 test 순위를 나란히 쓴다.

In [8]:
val_rank  = [r[0] for r in rows]                                   # val F1 오름... 내림순
test_rank = [r[0] for r in sorted(rows, key=lambda r: -f1_score(yte, r[2].predict(Xte_s)))]
print("val  F1 순위: " + " > ".join(val_rank))
print("test F1 순위: " + " > ".join(test_rank))
print()
print("이 데이터에서는 두 순위가 우연히 일치한다 — 그러나 이는 *보장이 아니다*.")
print("val 차이(여기서는 1위-2위가 ~0.014)가 val셋 잡음보다 작으면 순위가 뒤집힐 수 있다.")
print("§6의 E[max]가 바로 그 '뒤집힐 수 있는 폭'의 크기다.")

val  F1 순위: logreg(Ch02) > kNN k=5(Ch04) > RF(Ch07.2) > GBDT(Ch07.3) > tree d=3(Ch07.1)
test F1 순위: logreg(Ch02) > kNN k=5(Ch04) > RF(Ch07.2) > GBDT(Ch07.3) > tree d=3(Ch07.1)

이 데이터에서는 두 순위가 우연히 일치한다 — 그러나 이는 *보장이 아니다*.
val 차이(여기서는 1위-2위가 ~0.014)가 val셋 잡음보다 작으면 순위가 뒤집힐 수 있다.
§6의 E[max]가 바로 그 '뒤집힐 수 있는 폭'의 크기다.


In [9]:
# 5개 모델의 val F1 vs test F1을 나란히 (베이스라인 포함) — 본문 그림
names   = [r[0] for r in sorted(rows, key=lambda r: -r[1])]
val_f1s = [r[1] for r in sorted(rows, key=lambda r: -r[1])]
te_f1s  = [f1_score(yte, r[2].predict(Xte_s)) for r in sorted(rows, key=lambda r: -r[1])]
te_f1s  = te_f1s[:len(names)]
base    = [base_val_f1] * len(names)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
idx = np.arange(len(names))
ax.barh(idx + 0.19, val_f1s, height=0.34, label="val F1 (used for ranking)", color="#79a8d9")
ax.barh(idx - 0.19, te_f1s,  height=0.34, label="test F1 (measured once)", color="#4361ee")
ax.axvline(base_val_f1, color="#c0392b", ls="--", lw=1.4)
ax.text(base_val_f1, len(names)-0.42, f"baseline {base_val_f1:.3f}",
        color="#c0392b", fontsize=9, va="top")
ax.set_yticks(idx, names, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlim(0.72, 1.0)
ax.set_xlabel("F1")
ax.legend(fontsize=9, loc="lower right")
ax.set_title("val F1 vs test F1 — ranking decided by val only", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(IMG, "ch16_3_val_test_f1.svg"))
fig.show()
print("저장: ch16_3_val_test_f1.svg")

저장: ch16_3_val_test_f1.svg


## 8. 누수 시연: "해로운데, 깨끗한 데이터에선 안 보인다"

누수(Ch06.3)의 위험성은 **해로움을 증명할 수 없어서**가 아니라
**보이지 않아서**다. 두 가지로 보여준다.

**(a) 전처리 통계량 누수**: `StandardScaler`를 train+val+test **전체**로
fit. 이 잘 섞인 깨끗한 데이터에서는 train만 fit한 것과 스케일
통계량이 거의 같아, kNN val F1에 *가시적인 차이가 0*이다.

**(b) 그룹(환자) 누수**: train에 이미 있는 **같은 행**(= 같은 환자의
검사)을 "test"에 그대로 넣기. 모델도 하이퍼파라미터도 바꾸지 않았다 —
**분할 방식만** 바꿨을 뿐인데 "test 정확도"가 어떻게 되는가?

In [10]:
# (a) 전체 데이터로 fit한 스케일러
sc_all = StandardScaler().fit(X)
rel_diff = (np.abs(sc_all.scale_ - sc.scale_) / sc.scale_).max()
knn_all = KNeighborsClassifier(n_neighbors=5).fit(sc_all.transform(Xtr), ytr)
f1_all  = f1_score(yva, knn_all.predict(sc_all.transform(Xva)))
knn_tr  = KNeighborsClassifier(n_neighbors=5).fit(Xtr_s, ytr)
f1_tr   = f1_score(yva, knn_tr.predict(Xva_s))
print(f"(a) 스케일 표준편차 상대 최대 차이: {rel_diff:.3%}  ->  kNN val F1: 전체-fit {f1_all:.3f} vs train-fit {f1_tr:.3f}")
print("    잘 섞인 깨끗한 데이터에선 차이가 안 보임 — 이것이 '보이지 않는 누수'다.")
print()

# (b) 같은 환자(같은 행)를 test에 넣기
rng = np.random.default_rng(0)
idx = rng.choice(len(Xtr), 50, replace=False)
Xdup_s = Xtr_s[idx]                    # train의 50행을 'test'로
ydup   = ytr[idx]
knn1 = KNeighborsClassifier(n_neighbors=1).fit(Xtr_s, ytr)
leak_acc = knn1.score(Xdup_s, ydup)    # 같은 행이 train에 존재 -> 1.0
real_acc = knn1.score(Xte_s, yte)
print(f"(b) 누수된 'test' 정확도: {leak_acc:.3f}   실제 test 정확도(kNN-1): {real_acc:.3f}")
print(f"    차이 {leak_acc - real_acc:+.3f} — 모델/하이퍼파라미터 불변, '분할 방식만'")

(a) 스케일 표준편차 상대 최대 차이: 13.198%  ->  kNN val F1: 전체-fit 0.979 vs train-fit 0.979
    잘 섞인 깨끗한 데이터에선 차이가 안 보임 — 이것이 '보이지 않는 누수'다.

(b) 누수된 'test' 정확도: 1.000   실제 test 정확도(kNN-1): 0.947
    차이 +0.053 — 모델/하이퍼파라미터 불변, '분할 방식만'


## 9. 이 흐름이 기말 프로젝트(16.1~16.2)의 요구사항과 어떻게 대응되는가

| 16.1절 요구사항 | 이 노트북의 대응 |
|---|---|
| 문제 정의 (행·라벨·지표·베이스라인) | §1 + §4 |
| train/val/test 3분할, test 한 번만 (Ch06.3) | §3, §5 (+§6: "두 번 쓰면"의 정량적 대가) |
| 전처리 통계량 train fit (Ch06.3) | §2 (+§8a: 전체-fit이 *안 보임*) |
| val에서만 모델/하이퍼파라미터 결정 (Ch06.2) | §3 val F1 순위, §7 |
| Ch09~15 딥러닝 기법 ≥1 | (팀의 몫 — 이 노트북은 Block A 모델로 **동일한 절차**를 시연) |
| "왜 고전 ML이 아니라 딥러닝인가" (16.1 필수 질문) | §3: 고전 도구가 이미 0.97+이면, 딥러닝의 추가 근거를 *스스로* 만들어야 한다 |
| 리뷰어가 할 반박 가능한 질문 (16.2) | §6·§7·§8의 세 숫자 — "선택 편향/순위 뒤집기/누수" |

이 9단계가 곧 16.2절에서 다른 팀의 보고서를 볼 때 쓰는 **검토 눈**이다 —
특히 §8b처럼 "분할 방식" 하나에서 정확도가 +0.05 이상 움직인다면,
해당 팀에 던질 3점 질문은 이미 준비되어 있다.